<H3>Task: Object detecton on Indoor dataset</H3>


GPU check, clone and install libraries

In [ ]:
!nvidia-smi -L
!git clone https://github.com/anshugarg1/object_detection.git
%cd object_detection
!sed -i -E '/"torch(vision)?>=/d' pyproject.toml
!pip install -e . -q          

Download and unzip dataset

In [ ]:
!wget -q https://zenodo.org/records/2654485/files/Indoor%20Object%20Detection%20Dataset.zip -P data/raw/
!unzip -q data/raw/*.zip -d data/raw/
!mkdir -p "data/raw/Indoor_Object_Detection_Dataset"
!mv "data/raw/Indoor Object Detection Dataset"/* "data/raw/Indoor_Object_Detection_Dataset"/
!rm -rf "data/raw/Indoor Object Detection Dataset"

Pre-process and split dataset in yolo and coco format

In [ ]:
!python -m object_detection.data_preprocessing.convert   # dlib XML -> COCO + YOLO
!python -m object_detection.data_preprocessing.split     # stratified 80/10/10, prints per-class table

Train model or load pre-trained model weights

In [ ]:
from object_detection.modeling.train_yolo import train
from object_detection.config import PROJ_ROOT, REPORTS_DIR, RUN_NAME, MODELS_DIR

SKIP_TRAINING = True

weights = MODELS_DIR / f"{RUN_NAME}_best.pt"
if SKIP_TRAINING and weights.exists():
    print(f"Skipping training — using committed weights: {weights}")
else:
    weights = train()

print("Using weights:", weights)

Evaluate trained model on val and test dataset

In [ ]:
import yaml
import json
from object_detection.modeling.train_yolo import evaluate
cfg = yaml.safe_load(open(PROJ_ROOT/ "object_detection/configs/train_yolo.yaml"))
all_metrics = {s: evaluate(weights, s, cfg) for s in ("val", "test")}
out = REPORTS_DIR / f"metrics_{RUN_NAME}.json"
out.write_text(json.dumps(all_metrics, indent=2))

Display results

In [ ]:
from IPython.display import Markdown, display
from object_detection.config import REPORTS_DIR
display(Markdown((REPORTS_DIR / 'metrics_table.md').read_text()))

Best and worst detections

In [ ]:
from object_detection.plot import show_best_worst
from IPython.display import Image, display
out = show_best_worst('val', k=4)
display(Markdown("### Validation — best examples (green = prediction, red = ground truth)"))
for img in sorted(out.glob('*.jpg')):
    print(img.name); display(Image(str(img)))